
# AI in One Lab: End-of-Semester Survey (1.5 Hours)

This lab is designed as a **broad, hands-on tour** of several important AI ideas:

- Pandas data preprocessing  
- Logistic regression  
- Decision trees  
- Reinforcement learning (RL) — high-level demo  
- Retrieval-Augmented Generation (RAG) — tiny retrieval demo

You do **not** need deep prior AI experience. Work through the sections in order.

---

## Setup Instructions

1. Run the setup cell below first.  
2. Then work through Sections 1–5.  
3. At the end, answer the reflection questions in Section 6.


In [ ]:

# === SETUP: Install packages (Colab) ===
# This may take a couple minutes the first time.
!pip install -q pandas scikit-learn matplotlib sentence-transformers faiss-cpu gymnasium stable-baselines3

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5)

from sentence_transformers import SentenceTransformer
import numpy as np

import gymnasium as gym

from stable_baselines3 import PPO



---

## 1. Pandas Data Preprocessing (≈15 minutes)

We will use a small version of the **Titanic** dataset to practice:

- Loading a CSV with Pandas  
- Inspecting the data  
- Handling missing values  
- Encoding categorical features  
- Creating a train/test split


In [ ]:

# 1A. Load and inspect the dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("First few rows:")
display(df.head())

print("\nData info:")
df.info()


In [ ]:

# 1B. Select a subset of useful columns
# We'll try to predict whether a passenger survived based on class, sex, and age.
df = df[["Survived", "Pclass", "Sex", "Age"]]

print("After selecting subset:")
display(df.head())


In [ ]:

# 1C. Handle missing values
# Age has missing values; we'll fill with the median age.
median_age = df["Age"].median()
df["Age"] = df["Age"].fillna(median_age)

# 1D. Encode categorical 'Sex' as numeric: male -> 0, female -> 1
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

print("After cleaning and encoding:")
display(df.head())

print("\nCheck for remaining missing values:")
print(df.isna().sum())


In [ ]:

# 1E. Train/test split
X = df[["Pclass", "Sex", "Age"]]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)



---

## 2. Logistic Regression (≈15 minutes)

Now we'll train a simple **logistic regression classifier** to predict survival.

Steps:

1. Train the model on the training set.  
2. Evaluate accuracy on the test set.  
3. Look at the learned weights for each feature.


In [ ]:

# 2A. Train logistic regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

print("Model trained.")


In [ ]:

# 2B. Evaluate accuracy
y_pred_lr = log_reg.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Test Accuracy: {acc_lr:.3f}")


In [ ]:

# 2C. Inspect learned weights
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Weight": log_reg.coef_[0]
})
display(coef_df)



**Question (for you to think about):**  

- Which features seem to increase survival probability?  
- Which features seem to decrease it?



---

## 3. Decision Tree (≈15 minutes)

Next we will train a **decision tree classifier** on the same data and:

- Compare its accuracy to logistic regression  
- Visualize the tree to see how it splits the data


In [ ]:

# 3A. Train a decision tree
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf.fit(X_train, y_train)

y_pred_tree = tree_clf.predict(X_test)
acc_tree = accuracy_score(y_test, y_pred_tree)
print(f"Decision Tree Test Accuracy: {acc_tree:.3f}")


In [ ]:

# 3B. Visualize the tree
plt.figure(figsize=(12, 6))
plot_tree(
    tree_clf,
    feature_names=X.columns,
    class_names=["Died", "Survived"],
    filled=True,
    rounded=True
)
plt.title("Decision Tree (max_depth=3)")
plt.show()



**Questions:**  

1. How does the decision tree accuracy compare to logistic regression?  
2. From the plot, which feature appears at the top of the tree (root split)?  
3. Do the splits make intuitive sense?



---

## 4. Reinforcement Learning Demo (≈10–15 minutes)

Here we won’t train from scratch (that can take too long), but we will:

- Use an existing **CartPole** environment from `gymnasium`  
- Load a **pretrained PPO agent**  
- Watch it interact with the environment (in code) and see the reward loop

**Key Concepts:**

- **Environment**: provides state (observation), reward, and done flag  
- **Agent / Policy**: chooses actions based on observations  
- **Episode**: one run of the environment until it terminates  
- **Goal**: maximize cumulative reward


In [ ]:

# 4A. Create the environment
env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, info = env.reset()
print("Initial observation:", obs)


In [ ]:

# 4B. Download a pretrained PPO agent for CartPole
# (If this cell errors because of network issues, you can skip to the comments below.)
!wget -q -O ppo-CartPole-v1.zip https://huggingface.co/sb3/ppo-CartPole-v1/resolve/main/ppo-CartPole-v1.zip

# Load the model
rl_model = PPO.load("ppo-CartPole-v1")
print("Pretrained PPO model loaded.")


In [ ]:

# 4C. Run one episode with the pretrained agent
obs, info = env.reset()
total_reward = 0.0

for step in range(500):
    action, _ = rl_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

print(f"Episode finished after {step+1} steps.")
print(f"Total reward: {total_reward}")



**Questions:**  

1. What is the *state* in this environment?  
2. What is the *action* space?  
3. What is the *reward* signal encouraging the agent to do?  
4. How is this different from supervised learning (like logistic regression and decision trees)?



---

## 5. Mini RAG-Style Retrieval Demo (≈15 minutes)

In this section, we’ll simulate the **retrieval** part of Retrieval-Augmented Generation (RAG):

1. We define a small set of reference documents.  
2. We embed them into vectors using a **Sentence Transformer**.  
3. Given a user question, we retrieve the most similar document.

In a full RAG system, this retrieved document would be passed to a large language model (LLM) as context.


In [ ]:

# 5A. Tiny document collection
documents = [
    "The mitochondria is the powerhouse of the cell.",
    "Python is a programming language often used for machine learning.",
    "George Washington was the first President of the United States.",
    "Reinforcement learning trains agents using rewards and punishments.",
    "Decision trees split data based on features to make predictions.",
]

for i, doc in enumerate(documents):
    print(f"{i}: {doc}")


In [ ]:

# 5B. Load a sentence-transformer model and embed documents
rag_model = SentenceTransformer('all-MiniLM-L6-v2')

doc_embeddings = rag_model.encode(documents, normalize_embeddings=True)
doc_embeddings.shape


In [ ]:

# 5C. Simple retrieval function using cosine similarity (via dot product with normalized vectors)
def retrieve(query, k=1):
    q_emb = rag_model.encode([query], normalize_embeddings=True)[0]
    scores = np.dot(doc_embeddings, q_emb)
    idx = np.argsort(-scores)[:k]
    return idx, scores[idx]

# Try it with a question related to reinforcement learning
question = "How do agents learn from rewards in AI?"
idx, scores = retrieve(question, k=2)

print("Question:", question)
print("\nTop retrieved documents:")
for rank, (i, s) in enumerate(zip(idx, scores), start=1):
    print(f"{rank}. [score={s:.3f}] {documents[i]}")



**Try it yourself:** Change the question string above and see which document is retrieved.
Examples:
- `"What is Python used for?"`
- `"Who was the first US president?"`
- `"How do decision trees make predictions?"`



---

## 6. Reflection (≈10 minutes)

In your own words, answer the following questions (you can type answers in a new text cell below, or on a separate document as directed by your instructor):

1. **Data Preprocessing:** Why is data cleaning and preprocessing (e.g., handling missing values, encoding categories) important before training a model?  
2. **Model Choice:** In what situations might you prefer logistic regression over a decision tree, and vice versa?  
3. **Reinforcement Learning:** How is reinforcement learning conceptually different from supervised learning?  
4. **RAG:** What problem does retrieval-augmented generation (RAG) help solve compared to using a language model alone?  
5. **Next Steps:** Which of today’s topics (pandas, logistic regression, decision trees, RL, RAG) would you most like to explore in more depth, and why?



_Add your reflection answers here (or follow your instructor's submission instructions)._ 
